# Make the promise run
**IT 630 · October 21 · about 32 minutes · pairs welcome**

A pipeline completed. Which promises did its data keep?
You will inspect **five independent batches** from the same fictional dining hall.
No local installation, data download, dbt project or account beyond Colab is needed.

**Start:** File → Save a copy in Drive. Run cells with Shift+Enter.
Use the supplied setup. Fill the two SQL predicates before running all checks;
`___` deliberately stops execution until replaced. Write predictions **before** running.
Amounts and thresholds are authored teaching constants, not real dining policy.

**Hand in:** answer the text cells, then Share → Anyone with the link → Viewer.
Paste that Colab URL into Canvas; do not upload a notebook file.

## 1. Read the specimen (4 minutes)
Run setup, then calculate the clean total, reference mean and half-mean threshold yourself.
What does one row mean in this batch?

In [ ]:
import pandas as pd
import duckdb
from IPython.display import display
print("pandas", pd.__version__, "DuckDB", duckdb.__version__)
# Fictional H3 dining purchases: all amounts are intended USD dollars.
base = pd.DataFrame([
    ("R301", "H3", "2026-10-20T11:40:00-04:00", 4.50),
    ("R302", "H3", "2026-10-20T11:52:00-04:00", 6.00),
    ("R303", "H3", "2026-10-20T12:05:00-04:00", 5.25),
    ("R304", "H3", "2026-10-20T12:31:00-04:00", 7.00),
    ("R305", "H3", "2026-10-20T13:02:00-04:00", 3.25),
], columns=["purchase_id", "hall_id", "purchased_at", "amount"])
history = pd.DataFrame({"day": ["2026-10-15", "2026-10-16", "2026-10-17", "2026-10-18", "2026-10-19"],
                        "purchase_count": [5, 6, 5, 4, 5]})
batches = {
    "clean": base.copy(),
    "replay": pd.concat([base, base], ignore_index=True),
    "rename": base.rename(columns={"purchased_at": "timestamp"}),
    "cents": base.assign(amount=(base.amount * 100).round().astype(int)),
    "partial": base.iloc[:2].copy(),
}
display(base)
display(history)

**Your calculation and grain sentence:**

_Write here before continuing._

## 2. Make two promises executable (7 minutes)
Each check returns **violations**. Zero rows means this check found none.

| Check | Promise | Returned evidence |
|---|---|---|
| C1 | All four required column names exist; extra columns allowed | Missing names |
| C2 | purchase_id is nonnull and unique | Offending ID and its count |
| C3 | amount is nonnull and between 0.50 and 50.00 inclusive | Purchase ID and amount |
| C4 | Today's H3 count is at least half its reference mean | Actual count, reference mean, threshold |

C1 gates the run: if it fails, C2–C4 are **SKIPPED**, not passed.
Otherwise C2–C4 run independently, even when one fails.
C4 is intentionally **one-sided**; do not add an upper bound yet.

Complete C3's WHERE predicate (including nulls) and C4's WHERE predicate.
Keep the supplied result columns. Explain why WHERE selects bad rows rather than good ones.

In [ ]:
queries = {}
queries['C1'] = """
WITH required(column_name) AS (
 VALUES ('purchase_id'), ('hall_id'), ('purchased_at'), ('amount')
)
SELECT column_name AS missing_column FROM required
EXCEPT
SELECT column_name FROM information_schema.columns
WHERE table_name = 'purchases' AND table_schema = 'main'
"""

queries['C2'] = """
SELECT purchase_id, COUNT(*) AS occurrences
FROM purchases
GROUP BY purchase_id
HAVING purchase_id IS NULL OR COUNT(*) > 1
ORDER BY purchase_id
"""

queries['C3'] = """
SELECT purchase_id, amount FROM purchases
WHERE ___
ORDER BY purchase_id
"""

queries['C4'] = """
WITH actual AS (SELECT COUNT(*) AS actual_count FROM purchases),
reference AS (SELECT AVG(purchase_count) AS reference_mean FROM history)
SELECT actual_count, reference_mean, 0.5 * reference_mean AS threshold
FROM actual CROSS JOIN reference
WHERE ___
"""

**Why do the predicates select violations?** _Write here._

## 3. Predict, then run one batch at a time (12 minutes)
For each batch, record PASS / FAIL / SKIPPED for every check **before** running it.
Then run the next cell five times, changing `batch_name` each time.
Inspect the returned evidence; copy one diagnostic value for each failure.
Do not combine the batch changes.

| Batch | Your total (numeric amount sum) | C1 | C2 | C3 | C4 | Evidence |
|---|---|---|---|---|---|---|
| clean | | | | | | |
| replay: identical file appended twice | | | | | | |
| rename: purchased_at → timestamp | | | | | | |
| cents: numeric amount multiplied by 100 | | | | | | |
| partial: first two rows only | | | | | | |

The cents sum is a raw numeric sum; do not label it correct dollar revenue.
For rename, write “report not accepted” rather than a dollar report.
The schema gate is an activity policy, not a claim that other queries cannot run.

In [ ]:
def run_checks(batch):
    # Each call gets a new database, so batches never stack on each other.
    with duckdb.connect(":memory:") as con:
        con.register("incoming", batch)
        con.register("history_input", history)
        con.execute("CREATE TABLE purchases AS SELECT * FROM incoming")
        con.execute("CREATE TABLE history AS SELECT * FROM history_input")
        evidence = {"C1": con.execute(queries["C1"]).df()}
        if not evidence["C1"].empty:
            return {"C1": "FAIL", "C2": "SKIPPED", "C3": "SKIPPED", "C4": "SKIPPED"}, evidence
        for check in ("C2", "C3", "C4"):
            evidence[check] = con.execute(queries[check]).df()
        status = {check: "PASS" if rows.empty else "FAIL" for check, rows in evidence.items()}
        return status, evidence

In [ ]:
batch_name = "clean"  # Repeat with replay, rename, cents, partial.
status, evidence = run_checks(batches[batch_name])
print(batch_name, status)
for check, rows in evidence.items():
    print(check, "returned", len(rows), "violation record(s)")
    display(rows)
if status["C1"] == "PASS":
    print("Raw numeric amount sum:", round(float(batches[batch_name].amount.sum()), 2))

## 4. Explain what the checks miss (7 minutes)
1. Which check alone flags partial delivery? Why can complete, valid-looking rows still make an incomplete batch?
2. Consider **C4 alone** on the replay. Does it pass? Explain using the count and threshold. Does C2 provide different evidence?
3. A real $0.25 purchase arrives as numeric **25 cents**, but the reader treats it as dollars. Does **C3 alone** pass? What units/version evidence would you require? Would dividing every batch by 100 safely repair this?
4. Could a legitimate quiet day fail C4? Suggest a history-policy improvement and one risk it introduces. Do not change the assigned checks to make your original matrix look better.
5. Does a passing check repair data? Distinguish a warning/error policy from a repair.

**Your explanations:** _Write here._

**Exit:** In one sentence, what does a completed run establish, and what does a passing check establish?